# mSigLIP Colab Training Experiments

Notebook này gom các bước cần để chạy training experiments trên Google Colab Free khi `VN3K/` và `m_siglip_checkpoints/` đã có sẵn trên Google Drive.

Mặc định notebook dùng TensorBoard thay vì W&B để tránh bước login/secrets. Các lệnh full training dùng cấu hình memlite cho Colab Free: batch nhỏ hơn, accumulation lớn hơn, checkpoint lưu thẳng vào Drive.

## 0. Runtime Check

Chạy cell này trước để xác nhận Colab đang dùng GPU runtime. Nếu `torch.cuda.is_available()` là `False`, vào `Runtime > Change runtime type > GPU` rồi chạy lại notebook.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path


def run(cmd, check=True, cwd=None, env=None):
    print(f"$ {cmd}")
    return subprocess.run(cmd, shell=True, check=check, cwd=cwd, env=env)

run("nvidia-smi", check=False)

try:
    import torch
    print("Python:", sys.version)
    print("Torch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("CUDA:", torch.version.cuda)
        print("GPU:", torch.cuda.get_device_name(0))
except Exception as exc:
    print("Torch import failed:", repr(exc))

## 1. Mount Drive + Path Config

Sửa các biến dưới đây nếu Drive của bạn đặt repo/data/model ở chỗ khác.

Hai layout phổ biến:

1. Chuẩn hóa:
   - `/content/drive/MyDrive/data/raw/VN3K/`
   - `/content/drive/MyDrive/artifacts/models/pretrained/m_siglip_checkpoints/model.safetensors`
2. Đặt trực tiếp dưới MyDrive:
   - `/content/drive/MyDrive/VN3K/`
   - `/content/drive/MyDrive/m_siglip_checkpoints/model.safetensors`

In [ ]:
from pathlib import Path
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print("Drive mount skipped or unavailable:", repr(exc))

DRIVE_ROOT = Path('/content/drive/MyDrive')

# Edit these if needed.
PROJECT_DIR = DRIVE_ROOT / 'mSigLIP' / 'code'
DRIVE_DATA_ROOT = DRIVE_ROOT / 'data' / 'raw'
DRIVE_PRETRAINED_ROOT = DRIVE_ROOT / 'artifacts' / 'models' / 'pretrained'
DRIVE_ARTIFACTS_ROOT = DRIVE_ROOT / 'msiglip_colab' / 'artifacts'

# Fallback layout: uncomment if VN3K and m_siglip_checkpoints are directly under MyDrive.
# DRIVE_DATA_ROOT = DRIVE_ROOT
# DRIVE_PRETRAINED_ROOT = DRIVE_ROOT

os.environ['MSIGLIP_DATA_ROOT'] = str(DRIVE_DATA_ROOT)
os.environ['MSIGLIP_PRETRAINED_ROOT'] = str(DRIVE_PRETRAINED_ROOT)
os.environ['MSIGLIP_ARTIFACTS_ROOT'] = str(DRIVE_ARTIFACTS_ROOT)
DRIVE_ARTIFACTS_ROOT.mkdir(parents=True, exist_ok=True)

print('PROJECT_DIR           =', PROJECT_DIR)
print('MSIGLIP_DATA_ROOT     =', os.environ['MSIGLIP_DATA_ROOT'])
print('MSIGLIP_PRETRAINED_ROOT =', os.environ['MSIGLIP_PRETRAINED_ROOT'])
print('MSIGLIP_ARTIFACTS_ROOT  =', os.environ['MSIGLIP_ARTIFACTS_ROOT'])

## 2. Repo Setup

Cell này chuyển vào repo và cài package ở chế độ editable. Notebook dùng `pip`, không dùng `uv`, vì Colab đã có Python runtime riêng.

Nếu Colab báo thiếu package, giữ `INSTALL_MINIMAL_DEPS=True`. Nếu dependency đã đầy đủ và muốn nhanh hơn, đổi thành `False`.

In [ ]:
import os
import sys
from pathlib import Path

assert PROJECT_DIR.exists(), f"PROJECT_DIR does not exist: {PROJECT_DIR}"
os.chdir(PROJECT_DIR)
print('cwd =', Path.cwd())

INSTALL_MINIMAL_DEPS = True

if INSTALL_MINIMAL_DEPS:
    # Keep torch from the Colab runtime. Install only project-side dependencies commonly missing in Colab.
    run(f"{sys.executable} -m pip install -q -e . --no-deps")
    run(
        f"{sys.executable} -m pip install -q "
        "hydra-core omegaconf lightning loguru prettytable peft safetensors "
        "transformers sentencepiece ftfy tensorboard wandb scikit-learn scipy seaborn matplotlib nltk"
    )
else:
    run(f"{sys.executable} -m pip install -q -e . --no-deps")

# The text augmentation module initializes NLTK stopwords at import time.
try:
    import nltk
    nltk.download('stopwords', quiet=True)
    nltk.download('wordnet', quiet=True)
    nltk.download('omw-1.4', quiet=True)
except Exception as exc:
    print('NLTK data setup warning:', repr(exc))

# Make local src importable immediately in this kernel.
src_dir = str(PROJECT_DIR / 'src')
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

print('Setup complete')

## 3. Asset Verification

Fail sớm nếu data/model path sai. Cần thấy:

- `$MSIGLIP_DATA_ROOT/VN3K`
- `$MSIGLIP_PRETRAINED_ROOT/m_siglip_checkpoints/model.safetensors`

In [ ]:
from pathlib import Path
import os

DATA_ROOT = Path(os.environ['MSIGLIP_DATA_ROOT'])
PRETRAINED_ROOT = Path(os.environ['MSIGLIP_PRETRAINED_ROOT'])
ARTIFACTS_ROOT = Path(os.environ['MSIGLIP_ARTIFACTS_ROOT'])

vn3k_dir = DATA_ROOT / 'VN3K'
model_path = PRETRAINED_ROOT / 'm_siglip_checkpoints' / 'model.safetensors'

print('VN3K dir:', vn3k_dir)
print('Model:', model_path)
print('Artifacts:', ARTIFACTS_ROOT)

assert vn3k_dir.exists(), f"Missing VN3K dir: {vn3k_dir}"
assert model_path.exists(), f"Missing mSigLIP checkpoint: {model_path}"

annotation_files = sorted(vn3k_dir.glob('data_captions*.json'))
image_samples = sorted(vn3k_dir.rglob('*.jpg'))[:5] + sorted(vn3k_dir.rglob('*.png'))[:5]

print('Annotation files:')
for path in annotation_files[:10]:
    print(' -', path)
print('Image samples:')
for path in image_samples[:10]:
    print(' -', path)

assert annotation_files, f"No data_captions*.json found under {vn3k_dir}"
assert image_samples, f"No image samples found under {vn3k_dir}"

## 4. Smoke Tests

Chạy unit tests nhanh và một `fast_dev_run` trên subset nhỏ. Đây là bước bắt buộc trước full training để tránh mất runtime Colab vì sai path/config.

In [ ]:
run(f"{sys.executable} -m unittest tests/test_lora_configs.py")
run(f"{sys.executable} -m unittest tests/test_part_alignment_loss.py")

In [ ]:
smoke_cmd = " ".join([
    f"{sys.executable} trainer.py -cn cir_msiglip",
    "trainer.fast_dev_run=2",
    "dataset.proportion=0.02",
    "dataset.batch_size=8",
    "dataset.test_batch_size=16",
    "dataset.num_workers=2",
    "trainer.accumulate_grad_batches=9",
    "++trainer.precision=16-mixed",
    "logger.logger_type=tensorboard",
    "loss.NACIR=false",
    "+lora=attn_ffn_r16",
])
run(smoke_cmd)

## 5. Experiment Commands

Các cell dưới đây không tự chạy hàng loạt. Chạy từng experiment một để tránh hết runtime/GPU quota.

Mặc định Colab Free:

- `dataset.batch_size=8`
- `trainer.accumulate_grad_batches=9`
- `dataset.test_batch_size=16`
- `dataset.num_workers=2`
- `++trainer.precision=16-mixed`
- `logger.logger_type=tensorboard`

In [ ]:
COMMON_OVERRIDES = [
    "trainer.max_epochs=60",
    "dataset.batch_size=8",
    "dataset.test_batch_size=16",
    "dataset.num_workers=2",
    "trainer.accumulate_grad_batches=9",
    "++trainer.precision=16-mixed",
    "logger.logger_type=tensorboard",
    "optimizer=cir_test",
    "optimizer.param_groups.default.lr=1e-4",
    "loss.NACIR=false",
]


def trainer_cmd(*extra, ckpt_path=None):
    parts = [f"{sys.executable} trainer.py -cn cir_msiglip"]
    parts.extend(COMMON_OVERRIDES)
    parts.extend(extra)
    if ckpt_path:
        parts.append(f"ckpt_path={ckpt_path}")
    return " ".join(parts)

commands = {
    'baseline_lora_default_memlite': trainer_cmd('+lora=default'),
    'attn_ffn_r16_memlite': trainer_cmd('+lora=attn_ffn_r16'),
    'attn_ffn_r32_memlite': trainer_cmd('+lora=attn_ffn_r32'),
    'part_align_attn_ffn_r16': trainer_cmd('loss.PART_ALIGN=true', '+lora=attn_ffn_r16'),
    'pissa_attn_ffn_r32_optional': trainer_cmd('+lora=attn_ffn_r32_pissa'),
}

for name, cmd in commands.items():
    print(f"\n## {name}\n{cmd}")

### 5.1 Baseline LoRA cũ memlite

Dùng để tạo mốc Colab memlite, không so tuyệt đối với server baseline nếu batch/precision khác.

In [ ]:
run(commands['baseline_lora_default_memlite'])

### 5.2 LoRA attn+FFN r16 memlite

Experiment nên chạy đầu tiên cho hướng LoRA mới trên Colab Free.

In [ ]:
run(commands['attn_ffn_r16_memlite'])

### 5.3 LoRA attn+FFN r32 memlite

Chỉ chạy nếu `r16` ổn RAM. Nếu OOM, giữ `r16` làm nhánh Colab chính.

In [ ]:
run(commands['attn_ffn_r32_memlite'])

### 5.4 Part Align + LoRA attn+FFN r16

Chạy sau khi LoRA-only có baseline tương đối trên Colab.

In [ ]:
run(commands['part_align_attn_ffn_r16'])

### 5.5 Optional PiSSA

PiSSA có thể tốn thời gian khởi tạo hơn. Chỉ chạy nếu Colab còn runtime và r32 không OOM.

In [ ]:
run(commands['pissa_attn_ffn_r32_optional'])

## 6. Resume + Monitor

Dùng section này nếu Colab disconnect hoặc hết phiên. Checkpoint `last.ckpt` được lưu dưới Drive artifacts.

In [ ]:
from pathlib import Path
import os

runs_dir = Path(os.environ['MSIGLIP_ARTIFACTS_ROOT']) / 'training' / 'runs'
last_ckpts = sorted(runs_dir.glob('**/last.ckpt'), key=lambda p: p.stat().st_mtime if p.exists() else 0)

if last_ckpts:
    latest_ckpt = last_ckpts[-1]
    print('Latest last.ckpt:', latest_ckpt)
    print('\nResume attn_ffn_r16 command:\n')
    print(trainer_cmd('+lora=attn_ffn_r16', ckpt_path=str(latest_ckpt)))
else:
    latest_ckpt = None
    print('No last.ckpt found under', runs_dir)

In [ ]:
# Run only after the previous cell finds latest_ckpt.
if latest_ckpt is not None:
    run(trainer_cmd('+lora=attn_ffn_r16', ckpt_path=str(latest_ckpt)))

In [ ]:
# TensorBoard. If the magic has trouble with env vars, paste the printed path manually.
print('TensorBoard logdir:', Path(os.environ['MSIGLIP_ARTIFACTS_ROOT']) / 'training' / 'runs')

In [ ]:
%load_ext tensorboard
%tensorboard --logdir "$MSIGLIP_ARTIFACTS_ROOT/training/runs"

## 7. Result Collection

Cell này liệt kê log/config/checkpoint quan trọng để tải về hoặc ghi vào journal sau full run.

In [ ]:
from pathlib import Path
import os

root = Path(os.environ['MSIGLIP_ARTIFACTS_ROOT']) / 'training' / 'runs'
patterns = ['**/train.log', '**/.hydra/config.yaml', '**/checkpoints/*.ckpt', '**/events.out.tfevents.*']

for pattern in patterns:
    matches = sorted(root.glob(pattern), key=lambda p: p.stat().st_mtime if p.exists() else 0)
    print(f"\n# {pattern} ({len(matches)} files)")
    for path in matches[-20:]:
        print(path)

print('\nSau khi có full-run result, ghi metric vào docs/journal/[train]-YYYY-MM-DD.md theo policy repo.')